# Query to encyclopedia

Pipeline:

1. **pygetpapers** downloads Europe PMC full text (`PMC*/eupmc_result.json`, `fulltext.xml`).
2. **semantic_corpus** ingests that folder into a BAGIT corpus and writes a review table.
3. Mark papers `include` in the review JSON (or use every paper on the first pass).
4. **txt2phrases** builds a wordlist from the included full texts.
5. **encyclopedia** turns that wordlist into HTML, with optional Wikipedia enrichment.

On a local machine the sibling checkouts are `../pygetpapers`, `../semantic_corpus`, and this repo. In Colab the next cell clones them.

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone --depth 1 https://github.com/petermr/pygetpapers.git /content/pygetpapers
    !git clone --depth 1 https://github.com/semanticClimate/semantic_corpus.git /content/semantic_corpus
    !git clone --depth 1 https://github.com/petermr/encyclopedia.git /content/encyclopedia
    !pip install -q -e /content/pygetpapers -e /content/semantic_corpus -e /content/encyclopedia txt2phrases amilib
    ROOT = Path("/content/encyclopedia")
else:
    ROOT = Path.cwd()
    if not (ROOT / "encyclopedia").is_dir():
        ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ROOT

## Parameters

`STOP_AFTER = "wordlist"` stops before Wikipedia lookups. Set it to `"encyclopedia"` when you want HTML entries. `"review"` stops after the review table so you can mark includes first.

In [ ]:
QUERY = '("marine heatwave" OR "ocean heatwave") AND (current OR circulation)'
LIMIT = 5
MAX_TERMS = 20
STOP_AFTER = "wordlist"  # download | review | wordlist | encyclopedia
USE_ALL_PAPERS = True
ADD_WIKIPEDIA = False

WORK_DIR = ROOT / "temp" / "query_to_encyclopedia" / "ocean_heatwaves"
WORK_DIR

In [ ]:
from encyclopedia.pipeline.query_to_encyclopedia import run_query_to_encyclopedia

result = run_query_to_encyclopedia(
    query=QUERY,
    work_dir=WORK_DIR,
    limit=LIMIT,
    use_all_papers=USE_ALL_PAPERS,
    max_terms=MAX_TERMS,
    add_wikipedia=ADD_WIKIPEDIA,
    stop_after=STOP_AFTER,
    title="Ocean heatwaves",
)
result

## After review

Open `result.review_json`, set `review_status` to `include` for the papers you want, then rerun from the existing download. This cell does not call Europe PMC again.

In [ ]:
REVIEWED = run_query_to_encyclopedia(
    pygetpapers_dir=WORK_DIR / "pygetpapers",
    work_dir=WORK_DIR,
    review_table=WORK_DIR / "corpus" / "analysis" / "review" / "review_table.json",
    skip_ingest=True,
    use_all_papers=False,
    max_terms=MAX_TERMS,
    add_wikipedia=True,
    stop_after="encyclopedia",
    title="Ocean heatwaves",
    encyclopedia_html=WORK_DIR / "encyclopedia.html",
)
REVIEWED.encyclopedia_html